# llmcall — Usage Guide

A practical walkthrough of every capability in the `llmcall` package, from a one-liner text generation to structured data extraction from PDFs and images.

---

**Contents**
1. [Setup](#1-setup)
2. [Text Generation](#2-text-generation)
3. [Structured Generation](#3-structured-generation)
4. [Streaming](#4-streaming)
5. [Decision Making](#5-decision-making)
6. [Text Extraction](#6-text-extraction)
7. [PDF Extraction](#7-pdf-extraction)
8. [Image Extraction](#8-image-extraction)
9. [Async & Concurrency](#9-async--concurrency)

## 1. Setup

Install the package and set your API key. `llmcall` uses [LiteLLM](https://docs.litellm.ai/docs/providers) under the hood, so any provider supported by LiteLLM works — OpenAI, Anthropic, Google, Ollama, and more.

In [ ]:
# %pip install llmcall  # uncomment if running outside this repo
import os

# Set your key here or via a .env file / environment variable
os.environ["LLMCALL_API_KEY"] = "sk-..."          # required
os.environ["LLMCALL_MODEL"]   = "openai/gpt-4.1"   # default; change to any LiteLLM-supported model
# os.environ["LLMCALL_BASE_URL"] = "http://localhost:11434"  # for Ollama / local endpoints

In [ ]:
from llmcall import (
    generate, agenerate,
    generate_decision, agenerate_decision,
    extract, aextract,
    extract_pdf, aextract_pdf,
    extract_image, aextract_image,
)
from pydantic import BaseModel, Field
from typing import Optional

print("llmcall ready.")

---
## 2. Text Generation

The simplest possible call: give the model a prompt, get a string back.

In [ ]:
response = generate("What is the capital of France?")
print(response)

In [ ]:
# Use the `instructions` parameter to set a system prompt
response = generate(
    prompt="Explain recursion.",
    instructions="You are a patient tutor who explains concepts using short, concrete analogies. Keep answers under 3 sentences.",
)
print(response)

---
## 3. Structured Generation

Pass a Pydantic model as `output_schema` and get back a validated instance instead of a raw string.
The model **must** support structured outputs (e.g. `openai/gpt-4.1`, `anthropic/claude-sonnet-4-6`).

In [ ]:
class BookRecommendation(BaseModel):
    title: str
    author: str
    year: int
    one_line_summary: str
    tags: list[str]

book = generate(
    "Recommend one classic science-fiction novel.",
    output_schema=BookRecommendation,
)

print(f"{book.title} ({book.year}) by {book.author}")
print(book.one_line_summary)
print("Tags:", ", ".join(book.tags))

In [ ]:
# Nested schemas work too
class Character(BaseModel):
    name: str
    role: str
    motivation: str

class Story(BaseModel):
    title: str
    genre: str
    synopsis: str
    characters: list[Character]
    moral: Optional[str] = None

story = generate(
    "Write a short story about a robot who discovers music for the first time.",
    output_schema=Story,
)

print(f"** {story.title} ** [{story.genre}]")
print(story.synopsis)
print()
for c in story.characters:
    print(f"  • {c.name} ({c.role}): {c.motivation}")
if story.moral:
    print(f"\nMoral: {story.moral}")

---
## 4. Streaming

Set `stream=True` to receive tokens as they are generated. The call returns an iterator of string chunks — useful for printing to a terminal or pushing to a UI in real time.

> Streaming is not compatible with `output_schema` (structured JSON requires the full response).

In [ ]:
for chunk in generate("Write a haiku about distributed systems.", stream=True):
    print(chunk, end="", flush=True)
print()  # newline at the end

---
## 5. Decision Making

`generate_decision` asks the model to pick one option from a list given some context. The response is a structured `Decision` object with `selection` and `reason` fields — ideal for routing, classification, or control-flow branches.

In [ ]:
from llmcall import generate_decision

decision = generate_decision(
    prompt="A user submitted a support ticket: 'My invoice from last month shows the wrong amount and I was double-charged.'",
    options=["billing", "technical_support", "general_inquiry", "account_management"],
)

print("Routing to:", decision.selection)
print("Reason:   ", decision.reason)

In [ ]:
# Decision making with a custom system prompt
decision = generate_decision(
    prompt="sentiment: The product broke after one day and support never replied.",
    options=["positive", "neutral", "negative"],
    instructions="You are a sentiment classifier. Given a piece of customer feedback, classify its sentiment.",
)

print("Sentiment:", decision.selection)

---
## 6. Text Extraction

`extract` takes raw, unstructured text and maps it to a Pydantic schema. Think of it as a structured parsing layer powered by an LLM.

In [ ]:
class EmailMetadata(BaseModel):
    subject: str
    sender_name: Optional[str] = None
    topic: str
    sentiment: str
    action_required: bool
    action_description: Optional[str] = None

raw_email = """
From: James Okafor <j.okafor@acmecorp.com>
Date: Tuesday, 11 March 2026

Hi team,

Just a reminder that the Q1 budget report is due by end of day Friday.
Please make sure all department leads have submitted their figures to finance
before then — we cannot proceed with the board presentation without it.

Thanks,
James
"""

meta = extract(text=raw_email, output_schema=EmailMetadata)

print(f"Subject : {meta.subject}")
print(f"From    : {meta.sender_name}")
print(f"Topic   : {meta.topic}")
print(f"Sentiment: {meta.sentiment}")
print(f"Action? : {meta.action_required} — {meta.action_description}")

In [ ]:
# Extract multiple entities from a longer document
class Address(BaseModel):
    street: str
    city: str
    country: str
    postcode: Optional[str] = None

class Person(BaseModel):
    full_name: str
    email: Optional[str] = None
    phone: Optional[str] = None
    address: Optional[Address] = None

class ContractParties(BaseModel):
    client: Person
    vendor: Person
    contract_date: str
    contract_value_usd: Optional[float] = None

contract_text = """
SERVICE AGREEMENT dated 1 February 2026 between:

CLIENT: Priya Sharma, residing at 14 Elm Street, London, UK, EC1A 1BB.
Email: priya.sharma@example.com  Phone: +44 7700 900123

VENDOR: BuildRight Ltd, represented by Thomas Webb, 88 Commerce Ave,
Dublin, Ireland, D02 XY45. Email: t.webb@buildright.ie

Total contract value: USD 24,500.
"""

parties = extract(text=contract_text, output_schema=ContractParties)

print("Client :", parties.client.full_name, "—", parties.client.email)
print("Vendor :", parties.vendor.full_name, "—", parties.vendor.email)
print("Date   :", parties.contract_date)
print("Value  : USD", parties.contract_value_usd)

---
## 7. PDF Extraction

`extract_pdf` accepts a PDF from **three sources** — a URL, a local file path, or raw bytes — and returns a validated Pydantic model.

**Model requirement:** the configured model must support document/PDF input. Good choices:
- `openai/gpt-4.1`
- `anthropic/claude-sonnet-4-6`
- `google/gemini-3-flash-preview`

A clear `ValueError` is raised if the model lacks PDF support.

In [ ]:
# ── Schema ───────────────────────────────────────────────────────────────────
class ResearchPaper(BaseModel):
    title: str
    authors: list[str]
    abstract: str
    keywords: list[str]
    year: Optional[int] = None

# ── Source A: URL ─────────────────────────────────────────────────────────────
# Replace with any publicly accessible PDF URL
pdf_url = "https://arxiv.org/pdf/1706.03762"  # Attention Is All You Need

paper = extract_pdf(source=pdf_url, output_schema=ResearchPaper)

print(f"Title   : {paper.title}")
print(f"Authors : {', '.join(paper.authors)}")
print(f"Year    : {paper.year}")
print(f"Keywords: {', '.join(paper.keywords)}")
print()
print("Abstract:")
print(paper.abstract)

In [ ]:
# ── Source B: local file path ─────────────────────────────────────────────────
class Invoice(BaseModel):
    invoice_number: str
    vendor_name: str
    vendor_email: Optional[str] = None
    issue_date: str
    due_date: Optional[str] = None
    line_items: list[str]
    subtotal: float
    tax: Optional[float] = None
    total: float
    currency: str = "USD"

# invoice = extract_pdf(source="/path/to/invoice.pdf", output_schema=Invoice)
# print(f"Invoice #{invoice.invoice_number} from {invoice.vendor_name}")
# print(f"Total: {invoice.currency} {invoice.total}")
print("(Uncomment and set a real path to run this cell.)")

In [ ]:
# ── Source C: raw bytes ───────────────────────────────────────────────────────
# Useful when the PDF comes from an upload, an S3 stream, etc.

# with open("/path/to/report.pdf", "rb") as f:
#     pdf_bytes = f.read()
# result = extract_pdf(source=pdf_bytes, output_schema=Invoice)
print("(Uncomment and provide real bytes to run this cell.)")

### Real-world example — extracting structured data from a financial report PDF

In [ ]:
class FinancialHighlights(BaseModel):
    company_name: str
    reporting_period: str
    revenue_usd_millions: Optional[float] = None
    net_income_usd_millions: Optional[float] = None
    earnings_per_share: Optional[float] = None
    key_risks: list[str] = Field(default_factory=list)
    forward_guidance: Optional[str] = None

# Replace with a real annual-report PDF URL
# report_url = "https://example.com/annual-report-2025.pdf"
# highlights = extract_pdf(source=report_url, output_schema=FinancialHighlights)
# print(f"{highlights.company_name} — {highlights.reporting_period}")
# print(f"Revenue : ${highlights.revenue_usd_millions}M")
# print(f"Net Income: ${highlights.net_income_usd_millions}M")
# print("Key risks:")
# for r in highlights.key_risks:
#     print(f"  • {r}")
print("(Uncomment and provide a real PDF URL to run this cell.)")

---
## 8. Image Extraction

`extract_image` accepts an image from a **URL, local path, or raw bytes** and extracts structured data from it.
MIME type is auto-detected from the file extension; use `media_type=` to override when passing raw bytes.

**Model requirement:** the configured model must support vision. The same models that support PDF input also support images.

In [ ]:
class ImageDescription(BaseModel):
    scene_type: str
    main_subjects: list[str]
    dominant_colors: list[str]
    mood: str
    text_present: bool
    extracted_text: Optional[str] = None

# Public image URL
img_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/PNG_transparency_demonstration_1.png/280px-PNG_transparency_demonstration_1.png"

desc = extract_image(source=img_url, output_schema=ImageDescription)

print(f"Scene   : {desc.scene_type}")
print(f"Subjects: {', '.join(desc.main_subjects)}")
print(f"Colors  : {', '.join(desc.dominant_colors)}")
print(f"Mood    : {desc.mood}")
if desc.text_present:
    print(f"Text    : {desc.extracted_text}")

In [ ]:
# Receipt / document OCR-style extraction
class Receipt(BaseModel):
    store_name: str
    date: Optional[str] = None
    items: list[str]
    subtotal: Optional[float] = None
    tax: Optional[float] = None
    total: float
    payment_method: Optional[str] = None

# receipt = extract_image(source="/path/to/receipt.jpg", output_schema=Receipt)
# print(f"{receipt.store_name} — total: ${receipt.total}")
# for item in receipt.items:
#     print(f"  • {item}")
print("(Uncomment and set a real receipt image path to run this cell.)")

In [ ]:
# Passing raw bytes with an explicit MIME type
# with open("diagram.webp", "rb") as f:
#     img_bytes = f.read()
# result = extract_image(source=img_bytes, output_schema=ImageDescription, media_type="image/webp")
print("(Uncomment and provide real bytes to run this cell.)")

---
## 9. Async & Concurrency

Every function has an `a`-prefixed async counterpart: `agenerate`, `agenerate_decision`, `aextract`, `aextract_pdf`, `aextract_image`.

Use `asyncio.gather` to run multiple extractions in parallel — this is much faster than sequential calls when you have several documents to process.

In [ ]:
import asyncio

# ── Simple async generation ───────────────────────────────────────────────────
response = await agenerate("Name three programming paradigms in one sentence each.")
print(response)

In [ ]:
# ── Async streaming ───────────────────────────────────────────────────────────
async for chunk in await agenerate("Count from 1 to 5, one number per line.", stream=True):
    print(chunk, end="", flush=True)
print()

In [ ]:
# ── Parallel extraction from multiple sources ─────────────────────────────────
# Fire off multiple PDF/image extractions at the same time.

class PaperSummary(BaseModel):
    title: str
    one_sentence_summary: str
    year: Optional[int] = None

pdf_urls = [
    "https://arxiv.org/pdf/1706.03762",   # Attention Is All You Need
    "https://arxiv.org/pdf/1810.04805",   # BERT
]

summaries = await asyncio.gather(
    *[aextract_pdf(url, PaperSummary) for url in pdf_urls]
)

for s in summaries:
    print(f"[{s.year}] {s.title}")
    print(f"       {s.one_sentence_summary}")
    print()

In [ ]:
# ── Mix different async operations in one gather ──────────────────────────────
class Sentiment(BaseModel):
    label: str
    confidence: str

reviews = [
    "Absolutely love this product — changed my life!",
    "Mediocre at best. Fell apart after a week.",
    "Does exactly what it says on the tin. Good value.",
]

results = await asyncio.gather(
    *[aextract(text=r, output_schema=Sentiment) for r in reviews]
)

for review, result in zip(reviews, results):
    print(f"{result.label:10s} ({result.confidence}) — \"{review[:50]}...\"")

---

## Summary

| Function | Input | Returns | Async version |
|---|---|---|---|
| `generate(prompt)` | string | `str` | `agenerate` |
| `generate(prompt, output_schema=)` | string | `BaseModel` | `agenerate` |
| `generate(prompt, stream=True)` | string | `Iterator[str]` | `agenerate` → `AsyncIterator[str]` |
| `generate_decision(prompt, options)` | string + list | `Decision` | `agenerate_decision` |
| `extract(text, output_schema)` | string | `BaseModel` | `aextract` |
| `extract_pdf(source, output_schema)` | URL / path / bytes | `BaseModel` | `aextract_pdf` |
| `extract_image(source, output_schema)` | URL / path / bytes | `BaseModel` | `aextract_image` |

Configure the model, API key, and base URL via environment variables:

```
LLMCALL_API_KEY=sk-...
LLMCALL_MODEL=openai/gpt-4.1
LLMCALL_BASE_URL=          # optional, for local/proxy endpoints
```

See the [LiteLLM provider list](https://docs.litellm.ai/docs/providers) for all supported models.